In [1]:
# Persona extractor tailored for your Personachat (Persona + chat) and ESConv (dialog -> content)
# Save as persona_extractor_debug.py and run in Colab / notebook (upload files to /content)

import os, json, torch
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq
)
from datasets import Dataset
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

persona_file = "/content/personachat.json"
esconv_file  = "/content/ESConv.json"


Using device: cuda


In [2]:
# Load PersonaChat -> history -> persona
# -------------------------
def load_personachat_history2persona(path, max_examples=None):
    data = json.load(open(path, "r", encoding="utf-8"))
    examples = []
    for item in data:
        # Persona text: you have "Persona" key in your sample
        persona_text = ""
        # common keys in your sample: "Persona"
        for k in ["Persona", "persona", "personality"]:
            if k in item and item[k]:
                val = item[k]
                if isinstance(val, list):
                    persona_text = " | ".join([v.strip() for v in val if isinstance(v, str) and v.strip()])
                else:
                    persona_text = str(val).strip()
                break

        # chat: your sample has "chat" as newline-separated string
        chat = item.get("chat") or item.get("dialog") or item.get("dialogue") or item.get("utterances") or ""
        turns = []
        if isinstance(chat, str):
            turns = [t.strip() for t in chat.split("\n") if t.strip()]
        elif isinstance(chat, list):
            for u in chat:
                if isinstance(u, str):
                    turns.append(u.strip())
                elif isinstance(u, dict):
                    txt = u.get("content") or u.get("text") or u.get("utterance") or ""
                    if txt:
                        turns.append(str(txt).strip())

        if not persona_text or not turns:
            continue

        src = "history: " + " </s> ".join(turns)
        tgt = persona_text
        examples.append({"source": src, "target": tgt})
        if max_examples and len(examples) >= max_examples:
            break

    print(f"[INFO] Loaded {len(examples)} history->persona examples from {path}")
    return examples

In [3]:
# Quick check on PersonaChat loader
# -------------------------
persona_examples = load_personachat_history2persona(persona_file, max_examples=20000)
if len(persona_examples) == 0:
    raise SystemExit("ERROR: No history->persona examples found. Check your personachat.json keys (Persona/chat).")

print("\n=== SAMPLE TRAINING PAIRS (first 5) ===")
for i, ex in enumerate(persona_examples[:5]):
    print(f"\n[{i+1}] SOURCE (trunc): {ex['source'][:200]}")
    print(f"[{i+1}] TARGET (persona): {ex['target'][:200]}")

# Trim for quick runs (optional)
persona_examples = persona_examples[:15000]
n = len(persona_examples)
train_ds = Dataset.from_list(persona_examples[:int(0.9*n)])
val_ds   = Dataset.from_list(persona_examples[int(0.9*n):])

[INFO] Loaded 8939 history->persona examples from /content/personachat.json

=== SAMPLE TRAINING PAIRS (first 5) ===

[1] SOURCE (trunc): history: hi , how are you doing ? i am getting ready to do some cheetah chasing to stay in shape . </s> you must be very fast . hunting is one of my favorite hobbies . </s> i am ! for my hobby i like 
[1] TARGET (persona): i like to remodel homes. i like to go hunting. i like to shoot a bow. my favorite holiday is halloween.

[2] SOURCE (trunc): history: hi , how are you doing today ? </s> i am spending time with my 4 sisters what are you up to </s> wow , four sisters . just watching game of thrones . </s> that is a good show i watch that whi
[2] TARGET (persona): my mom is my best friend. i have four sisters. i believe that mermaids are real. i love iced tea.

[3] SOURCE (trunc): history: we all live in a yellow submarine , a yellow submarine . morning ! </s> hi ! that is a great line for my next stand up . </s> lol . i am shy , anything to break th

In [4]:
# Model & Tokenization
# -------------------------
model_name = "facebook/bart-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

def tokenize_fn(batch):
    model_inputs = tokenizer(batch["source"], truncation=True, max_length=512)
    # correct seq2seq target tokenization
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(batch["target"], truncation=True, max_length=128)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Inspect tokenized labels for a small sample to detect -100 or empty targets
sample_check = Dataset.from_list(persona_examples[:4])
sample_tok = sample_check.map(tokenize_fn, batched=True, remove_columns=sample_check.column_names)
print("\n=== TOKENIZED LABELS SAMPLE ===")
for i, item in enumerate(sample_tok):
    labels = item["labels"]
    print(f"Example {i+1}: label tokens len={len(labels)} preview={labels[:20]}")
    if len(labels) == 0 or all([lbl == -100 for lbl in labels]):
        raise SystemExit("ERROR: tokenized labels appear empty or all -100. That means target tokenization failed or target is empty.")

train_tokenized = train_ds.map(tokenize_fn, batched=True, remove_columns=train_ds.column_names)
val_tokenized   = val_ds.map(tokenize_fn,   batched=True, remove_columns=val_ds.column_names)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Map:   0%|          | 0/4 [00:00<?, ? examples/s]


=== TOKENIZED LABELS SAMPLE ===
Example 1: label tokens len=30 preview=[0, 118, 101, 7, 21054, 523, 1611, 4, 939, 101, 7, 213, 8217, 4, 939, 101, 7, 4511, 10, 7323]
Example 2: label tokens len=29 preview=[0, 4783, 3795, 16, 127, 275, 1441, 4, 939, 33, 237, 7502, 4, 939, 679, 14, 9374, 16355, 29, 32]
Example 3: label tokens len=46 preview=[0, 118, 56, 10, 10196, 23, 400, 7364, 94, 363, 4, 939, 173, 25, 10, 1413, 62, 10688, 4, 939]
Example 4: label tokens len=25 preview=[0, 118, 524, 182, 7714, 4, 939, 3568, 9872, 4, 939, 33, 6219, 2549, 4, 939, 657, 44821, 21050, 4]


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:4034: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Map:   0%|          | 0/8045 [00:00<?, ? examples/s]

Map:   0%|          | 0/894 [00:00<?, ? examples/s]

In [5]:
# Sanity generation BEFORE training (should produce something but likely not persona-yet)
# -------------------------
def generate_from_text(inp_text, max_len=128):
    inputs = tokenizer(inp_text, return_tensors="pt", truncation=True, max_length=512).to(device)
    model.eval()
    with torch.no_grad():
        ids = model.generate(**inputs, max_length=max_len, num_beams=4, early_stopping=True, min_length=1)
    return tokenizer.decode(ids[0], skip_special_tokens=True).strip()

print("\n=== SANITY GENERATE (pre-train) on first source ===")
test_src = persona_examples[0]["source"]
print("SOURCE (trunc):", test_src[:200])
print("MODEL OUTPUT (pre-train):", repr(generate_from_text(test_src)))


=== SANITY GENERATE (pre-train) on first source ===
SOURCE (trunc): history: hi , how are you doing ? i am getting ready to do some cheetah chasing to stay in shape . </s> you must be very fast . hunting is one of my favorite hobbies . </s> i am ! for my hobby i like 
MODEL OUTPUT (pre-train): 'history: hi , how are you doing ? i am getting ready to do some cheetah chasing to stay in shape .'


In [6]:
# ===================================================
# TRAIN ON FULL PERSONACHAT → TEST ON FULL ESConv
# ===================================================

import json
import torch
from tqdm import tqdm
from datasets import Dataset
from transformers import (
    BartForConditionalGeneration,
    BartTokenizer,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq
)

device = "cuda" if torch.cuda.is_available() else "cpu"

# -----------------------------
# 1. LOAD PERSONACHAT DATA
# -----------------------------
personachat_path = "/content/personachat.json"
with open(personachat_path, "r", encoding="utf-8") as f:
    persona_raw = json.load(f)

print("Loaded PersonaChat entries:", len(persona_raw))

# Each entry has keys: "", "Persona", "chat"
def process_personachat_entry(entry):
    persona = entry.get("Persona", "").strip()
    chat_text = entry.get("chat", "").strip()
    if not persona or not chat_text:
        return None

    # Split chat into utterances
    turns = [t.strip() for t in chat_text.split("\n") if t.strip()]
    if len(turns) < 2:
        return None

    # Input = Persona + conversation history (except last)
    # Target = last utterance
    src = f"persona: {persona}  history: {' </s> '.join(turns[:-1])}"
    tgt = turns[-1]
    return {"source": src, "target": tgt}

processed_data = [d for d in (process_personachat_entry(x) for x in persona_raw) if d]
print("Processed PersonaChat samples:", len(processed_data))

dataset = Dataset.from_list(processed_data)

# -----------------------------
# 2. LOAD MODEL & TOKENIZER
# -----------------------------
model_name = "facebook/bart-base"
tokenizer = BartTokenizer.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name)

# -----------------------------
# 3. TOKENIZE
# -----------------------------
def preprocess_function(examples):
    model_inputs = tokenizer(
        examples["source"], truncation=True, padding="max_length", max_length=512
    )
    labels = tokenizer(
        examples["target"], truncation=True, padding="max_length", max_length=128
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized = dataset.map(preprocess_function, batched=True, remove_columns=dataset.column_names)

# -----------------------------
# 4. TRAINING
# -----------------------------
args = Seq2SeqTrainingArguments(
    output_dir="./persona_bart_full",
    per_device_train_batch_size=4,
    num_train_epochs=3,
    logging_steps=100,
    save_steps=500,
    save_total_limit=2,
    report_to="none",
    remove_unused_columns=False
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

print("\n=== START TRAINING ON FULL PERSONACHAT ===")
trainer.train()
trainer.save_model("./persona_bart_full")
tokenizer.save_pretrained("./persona_bart_full")
print("Training complete and model saved.")

# ===================================================
# 5. INFERENCE ON FULL ESConv
# ===================================================

esconv_path = "/content/ESConv.json"
with open(esconv_path, "r", encoding="utf-8") as f:
    esconv_data = json.load(f)

print("Loaded ESConv dialogues:", len(esconv_data))
model.to(device)
model.eval()

def extract_esconv_texts(dialog):
    """Extract the dialogue utterances from ESConv entry."""
    utterances = []
    if isinstance(dialog, dict) and "dialog" in dialog:
        for turn in dialog["dialog"]:
            content = turn.get("content", "").strip()
            if content:
                utterances.append(content)
    return utterances

def compose_input(history_texts):
    return "history: " + " </s> ".join(history_texts)

def infer_persona(dialog_entry):
    utterances = extract_esconv_texts(dialog_entry)
    if not utterances:
        return ""

    inp = compose_input(utterances)
    inputs = tokenizer(inp, return_tensors="pt", truncation=True, max_length=512).to(device)
    with torch.no_grad():
        ids = model.generate(**inputs, max_length=128, num_beams=4, early_stopping=True)
    return tokenizer.decode(ids[0], skip_special_tokens=True).strip()

output = []
for dlg in tqdm(esconv_data, desc="Generating personas from ESConv"):
    persona_pred = infer_persona(dlg)
    output.append({"dialog": dlg, "generated_persona": persona_pred})

with open("PESConv_full.json", "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print("PESConv_full.json saved.")


Loaded PersonaChat entries: 8939
Processed PersonaChat samples: 8939


Map:   0%|          | 0/8939 [00:00<?, ? examples/s]

/tmp/ipython-input-3435098932.py:89: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(



=== START TRAINING ON FULL PERSONACHAT ===


Step,Training Loss
100,3.045600
200,0.396400
300,0.370600
400,0.343200
500,0.354200
600,0.352500
700,0.339100
800,0.348800
900,0.339300
1000,0.342000


/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


Training complete and model saved.
Loaded ESConv dialogues: 1300


Generating personas from ESConv: 100%|██████████| 1300/1300 [03:09<00:00,  6.85it/s]


PESConv_full.json saved.
